In [1]:
import pandas as pd, numpy as np, json, os, plotly.express as px, plotly.io as pio
from pathlib import Path
from jinja2 import Environment

data_dir    = Path().resolve().parent / 'data'
outputs_dir = Path().resolve().parent / 'outputs'
os.makedirs(outputs_dir, exist_ok=True)

scored   = pd.read_csv(outputs_dir / 'scored_members.csv')
summary  = json.load(open(outputs_dir / 'scoring_summary.json'))
metrics  = json.load(open(outputs_dir / 'eval_metrics.json'))
feat_imp = pd.read_csv(outputs_dir / 'feature_importance.csv').head(10)
scored['condition_cluster'] = scored['condition_cluster'].astype(str)

COLOURS = {'High': '#E8533B', 'Medium': '#F2A65A', 'Low': '#4A6FA5'}

print(f"Loaded {len(scored):,} members for report generation")
print(f"  Nudge cohort: {scored['nudge_signal'].sum():,}")
print(f"  Precision@top20%: {metrics['precision_top20pct']:.3f}")
print(f"  ROC-AUC: {metrics['roc_auc']:.4f}")

Loaded 50,000 members for report generation
  Nudge cohort: 4,758
  Precision@top20%: 0.393
  ROC-AUC: 0.7417


In [2]:
# ── KPI calculations ──────────────────────────────────────────────────────────
total      = len(scored)
high_risk  = (scored['risk_tier'] == 'High').sum()
nudge_count = int(scored['nudge_signal'].sum())
nudge_rate  = scored['nudge_signal'].mean()
precision   = metrics['precision_top20pct']
true_pos    = round(nudge_count * precision)
conversions  = round(true_pos * 0.20)
cost_avoidance  = conversions * 800
protection_ratio = 1.88  # from DGP verification: allied health users have 1.88x lower acute risk

print(f"Total members:           {total:,}")
print(f"High risk:               {high_risk:,}  ({high_risk/total:.1%})")
print(f"Nudge candidates:        {nudge_count:,}  ({nudge_rate:.1%})")
print(f"Est. true positives:     {true_pos:,}")
print(f"Est. conversions (20%):  {conversions:,}")
print(f"Est. claims savings:     AUD ${cost_avoidance:,}")
print(f"Protection ratio:        {protection_ratio}x")

Total members:           50,000
High risk:               5,000  (10.0%)
Nudge candidates:        4,758  (9.5%)
Est. true positives:     1,870
Est. conversions (20%):  374
Est. claims savings:     AUD $299,200
Protection ratio:        1.88x


In [3]:
# ── Chart 1: Risk tier donut ──────────────────────────────────────────────────
tier_counts = scored['risk_tier'].value_counts().reindex(['High','Medium','Low'])
fig_tier = px.pie(
    values=tier_counts.values, names=tier_counts.index, hole=0.55,
    color=tier_counts.index, color_discrete_map=COLOURS,
    title='Population Risk Distribution'
)
fig_tier.update_traces(textposition='outside', textinfo='percent+label')
chart_tier = pio.to_html(fig_tier, full_html=False, include_plotlyjs='cdn')

In [4]:
# ── Chart 2: Risk score distribution ──────────────────────────────────────────
fig_dist = px.histogram(
    scored, x='risk_score', nbins=40, color='risk_tier',
    color_discrete_map=COLOURS, title='Risk Score Distribution',
    barmode='overlay', opacity=0.7
)
chart_dist = pio.to_html(fig_dist, full_html=False, include_plotlyjs=False)

In [5]:
# ── Chart 3: Nudge rate by condition cluster ──────────────────────────────────
nudge_by_cluster = (
    scored.groupby('condition_cluster')['nudge_signal']
    .agg(['sum','count'])
    .assign(rate=lambda x: x['sum']/x['count'])
    .sort_values('rate', ascending=False)
    .reset_index()
)
fig_cluster = px.bar(
    nudge_by_cluster, x='condition_cluster', y='rate',
    title='Nudge-Eligible Rate by Condition Cluster',
    color='rate', color_continuous_scale=['#4A6FA5','#E8533B'],
    text=nudge_by_cluster['rate'].map('{:.1%}'.format)
)
fig_cluster.update_traces(textposition='outside')
fig_cluster.update_layout(coloraxis_showscale=False)
chart_cluster = pio.to_html(fig_cluster, full_html=False, include_plotlyjs=False)

In [6]:
# ── Chart 4: Recommended modality (nudge cohort) ──────────────────────────────
nudge_df   = scored[scored['nudge_signal'] == 1]
mod_counts = nudge_df['recommended_modality'].value_counts().reset_index()
mod_counts.columns = ['modality','count']
fig_mod = px.pie(
    mod_counts, values='count', names='modality', hole=0.5,
    title='Recommended Coverage by Modality (Nudge Cohort)',
    color_discrete_sequence=['#00b894','#4A6FA5','#F2A65A','#E8533B']
)
fig_mod.update_traces(textposition='outside', textinfo='percent+label')
chart_mod = pio.to_html(fig_mod, full_html=False, include_plotlyjs=False)

In [7]:
# ── Chart 5: Feature importance (SHAP) ─────────────────────────────────────────
fig_shap = px.bar(
    feat_imp.sort_values('mean_abs_shap'),
    x='mean_abs_shap', y='feature', orientation='h',
    title='What Drives Acute Event Risk',
    color='mean_abs_shap', color_continuous_scale=['#4A6FA5','#E8533B']
)
fig_shap.update_layout(coloraxis_showscale=False)
chart_shap = pio.to_html(fig_shap, full_html=False, include_plotlyjs=False)

In [8]:
# ── Chart 6: Within-cluster tier comparison ────────────────────────────────────
# Show that every cluster now has High tier representation (cluster_tier fix)
if 'cluster_tier' in scored.columns:
    ct = scored.groupby('condition_cluster')['cluster_tier'].value_counts().unstack().fillna(0).astype(int)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    fig_ct = px.bar(
        ct_pct.reset_index(), x='condition_cluster', y=['High','Medium','Low'],
        title='Every Condition Type Gets Fair Attention',
        color_discrete_map=COLOURS, barmode='stack',
        labels={'value':'% of Cluster','condition_cluster':'Condition Cluster','variable':'Tier'}
    )
    chart_ct = pio.to_html(fig_ct, full_html=False, include_plotlyjs=False)
    print("cluster_tier chart generated")
else:
    chart_ct = ''
    print("cluster_tier column not found — skipping chart")

cluster_tier chart generated


In [9]:
# ── Business case sensitivity table ────────────────────────────────────────────
conv_rates = [0.10, 0.15, 0.20, 0.25, 0.30]
ed_costs   = [600, 800, 1000, 1200]
sensitivity = []
for cr in conv_rates:
    row = {'Conversion Rate': f'{cr:.0%}'}
    for ec in ed_costs:
        savings = round(true_pos * cr * ec)
        row[f'AUD ${ec}'] = f'${savings:,}'
    sensitivity.append(row)
sens_df = pd.DataFrame(sensitivity)
print(sens_df.to_string(index=False))

Conversion Rate AUD $600 AUD $800 AUD $1000 AUD $1200
            10% $112,200 $149,600  $187,000  $224,400
            15% $168,300 $224,400  $280,500  $336,600
            20% $224,400 $299,200  $374,000  $448,800
            25% $280,500 $374,000  $467,500  $561,000
            30% $336,600 $448,800  $561,000  $673,200


In [10]:
# ── Load causal chain SVG (optional) ───────────────────────────────────────────
causal_svg_path = outputs_dir / 'causal_chain.svg'
causal_chain_svg = ''
if causal_svg_path.exists():
    with open(causal_svg_path, encoding='utf-8') as f:
        causal_chain_svg = f.read()
    print(f"Causal chain SVG loaded ({causal_svg_path.stat().st_size} bytes)")
else:
    print("causal_chain.svg not found — causal chain section will be text-only")

# ── HTML Template ──────────────────────────────────────────────────────────────
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Allied Health Coverage — Claims Impact Analysis</title>
<style>
  body       { font-family: 'Segoe UI', Arial, sans-serif; margin: 0; background: #f5f6fa; color: #2d3436; }
  .header    { background: linear-gradient(135deg, #2d3436 0%, #636e72 100%); color: white; padding: 40px 60px 30px; }
  .header h1 { margin: 0 0 8px; font-size: 28px; font-weight: 600; }
  .header p  { margin: 0; opacity: 0.7; font-size: 14px; }
  .kpis      { display: flex; gap: 20px; padding: 30px 60px 10px; flex-wrap: wrap; }
  .kpi       { background: white; border-radius: 10px; padding: 24px 32px; flex: 1; min-width: 160px;
               box-shadow: 0 2px 8px rgba(0,0,0,0.07); border-left: 4px solid #4A6FA5; }
  .kpi.alert { border-left-color: #E8533B; }
  .kpi.warn  { border-left-color: #F2A65A; }
  .kpi.good  { border-left-color: #00b894; }
  .kpi h2    { margin: 0 0 6px; font-size: 32px; font-weight: 700; }
  .kpi p     { margin: 0; font-size: 13px; color: #636e72; }
  .section   { padding: 10px 60px 30px; }
  .section h2{ font-size: 18px; font-weight: 600; margin: 30px 0 12px;
               border-bottom: 2px solid #e0e0e0; padding-bottom: 6px; }
  .causal    { background: white; border-radius: 10px; padding: 24px 32px;
               box-shadow: 0 2px 8px rgba(0,0,0,0.07); margin-bottom: 24px; overflow-x: auto; }
  .causal h3 { margin: 0 0 16px; font-size: 15px; color: #636e72; font-weight: 600; }
  .charts    { display: grid; grid-template-columns: 1fr 1fr; gap: 24px; }
  .chart     { background: white; border-radius: 10px; padding: 16px;
               box-shadow: 0 2px 8px rgba(0,0,0,0.07); }
  .chart.wide{ grid-column: 1 / -1; }
  .bizcase   { background: white; border-radius: 10px; padding: 24px 32px;
               box-shadow: 0 2px 8px rgba(0,0,0,0.07); border-left: 4px solid #00b894; }
  .bizcase h3{ margin: 0 0 12px; font-size: 15px; font-weight: 600; color: #00b894; }
  .biz-row   { display: flex; gap: 32px; margin: 16px 0; flex-wrap: wrap; }
  .biz-num   { text-align: center; }
  .biz-num h3{ font-size: 28px; font-weight: 700; margin: 0; color: #2d3436; }
  .biz-num p { font-size: 12px; color: #636e72; margin: 4px 0 0; }
  .biz-note  { font-size: 12px; color: #636e72; margin-top: 12px; line-height: 1.6; }
  .evidence  { background: white; border-radius: 10px; padding: 24px 32px;
               box-shadow: 0 2px 8px rgba(0,0,0,0.07); border-left: 4px solid #4A6FA5; }
  .evidence h3{ margin: 0 0 12px; font-size: 15px; font-weight: 600; color: #4A6FA5; }
  .model     { background: white; border-radius: 10px; padding: 24px 32px;
               box-shadow: 0 2px 8px rgba(0,0,0,0.07); }
  .metrics   { display: flex; gap: 32px; margin: 16px 0; flex-wrap: wrap; }
  .metric    { text-align: center; }
  .metric h3 { font-size: 28px; font-weight: 700; margin: 0; color: #4A6FA5; }
  .metric p  { font-size: 12px; color: #636e72; margin: 4px 0 0; }
  table      { width: 100%; border-collapse: collapse; margin: 16px 0; font-size: 13px; }
  th, td     { padding: 8px 12px; text-align: right; border-bottom: 1px solid #e0e0e0; }
  th         { background: #f5f6fa; font-weight: 600; color: #2d3436; }
  th:first-child, td:first-child { text-align: left; }
  footer     { text-align: center; padding: 30px; color: #b2bec3; font-size: 12px; }
</style>
</head>
<body>
<div class="header">
  <h1>Allied Health Coverage — Claims Impact Analysis</h1>
  <p>{{ total_scored }} members analysed &middot; Proof of concept &middot; Synthetic dataset</p>
</div>

<div class="kpis">
  <div class="kpi"><h2>{{ total_scored }}</h2><p>Members Analysed</p></div>
  <div class="kpi alert"><h2>{{ high_risk }}</h2><p>High Risk Without Coverage</p></div>
  <div class="kpi warn"><h2>{{ nudge_count }}</h2><p>Coverage Candidates</p></div>
  <div class="kpi"><h2>{{ nudge_rate }}</h2><p>Coverage Rate</p></div>
  <div class="kpi good"><h2>{{ cost_avoidance }}</h2><p>Est. Claims Savings (AUD)<br><small>@ 20% conversion, $800/ED visit</small></p></div>
</div>

<div class="section">

  {% if causal_chain_svg %}
  <h2>Why Coverage Matters — The Causal Chain</h2>
  <div class="causal">
    <h3>When members don't use their allied health benefits, conditions escalate to avoidable ED visits</h3>
    {{ causal_chain_svg }}
    <p style="font-size:12px;color:#636e72;margin-top:12px;">
      Covering allied health (green) breaks the chain before escalation reaches the ED.
      Each outreach targets a member with a High or Medium risk score <strong>and</strong>
      unused benefits — every nudge is a concrete, costable intervention.
    </p>
  </div>
  {% else %}
  <h2>Why Coverage Matters</h2>
  <div class="causal">
    <h3>Unmanaged conditions escalate when members don't use their allied health benefits</h3>
    <p style="font-size:13px;color:#636e72;line-height:1.6;">
      <strong>Unmanaged condition</strong> → <strong>No allied health use</strong> →
      <strong>Condition escalates</strong> → <strong>ED visit / admission (avoidable cost)</strong>
      <br><br>
      <span style="color:#00b894;font-weight:600;">→ Nudge intervention: Use unused benefits → Cost avoided</span>
    </p>
  </div>
  {% endif %}

  <h2>Population Risk Distribution</h2>
  <div class="charts">
    <div class="chart">{{ charts['tier'] }}</div>
    <div class="chart">{{ charts['dist'] }}</div>
  </div>

  <h2>Coverage Candidates</h2>
  <div class="charts">
    <div class="chart">{{ charts['cluster'] }}</div>
    <div class="chart">{{ charts['modality'] }}</div>
  </div>

  {% if charts['ct'] %}
  <h2>Before &amp; After — Fair Tiering Across Conditions</h2>
  <div class="charts">
    <div class="chart wide">{{ charts['ct'] }}</div>
  </div>
  <p style="font-size:12px;color:#636e72;margin-top:4px;">
    Every condition cluster now has 10% High / 10% Medium within its own peer group.
    No cluster is overlooked — MSK, MH, Metabolic, Mixed, and Healthy all get
    proportional attention from the coverage programme.
  </p>
  {% endif %}

  <h2>The Evidence — Why Coverage Reduces Claims</h2>
  <div class="evidence">
    <h3>Two independent lines of evidence</h3>
    <div class="biz-row">
      <div class="biz-num">
        <h3>{{ protection_ratio }}x</h3>
        <p>Protection Ratio<br><small>Members using allied health have<br>{{ protection_ratio }}x lower acute event risk<br>(DGP-verified)</small></p>
      </div>
      <div class="biz-num">
        <h3>#3</h3>
        <p>SHAP Feature Rank<br><small>allied_health_utilisation_rate is the<br>#3 strongest predictor of acute risk<br>(SHAP mean |value| = 0.148)</small></p>
      </div>
      <div class="biz-num">
        <h3>{{ precision_pct }}%</h3>
        <p>Precision @ Top 20%<br><small>Of the highest-scored members,<br>{{ precision_pct }}% are true acute risks<br>(held-out test set)</small></p>
      </div>
      <div class="biz-num">
        <h3>{{ ndcg_5 }}</h3>
        <p>NDCG@5%<br><small>Ranking quality in the top 5%<br>of the scored list — where<br>nudge ROI is highest</small></p>
      </div>
    </div>
    <p class="biz-note">
      Allied health utilisation is the #3 strongest predictor of acute event risk (mean |SHAP| = 0.148),
      behind only comorbidity count (0.493) and condition cluster (0.169). In the underlying data
      generation process, members who use allied health have {{ protection_ratio }}x lower acute risk —
      the model has independently learned this signal from claims data alone. Coverage reduces claims:
      every nudge that converts is a prevented ED visit.
    </p>
  </div>

  <h2>What Drives Risk — SHAP Feature Importance</h2>
  <div class="charts"><div class="chart wide">{{ charts['shap'] }}</div></div>

  <h2>Business Case — Claims Savings Sensitivity</h2>
  <div class="bizcase">
    <h3>Estimated intervention value</h3>
    <div class="biz-row">
      <div class="biz-num"><h3>{{ nudge_count }}</h3><p>Coverage Candidates</p></div>
      <div class="biz-num"><h3>{{ true_positives }}</h3><p>Est. True Positives<br><small>(candidates &times; {{ precision_pct }}% precision)</small></p></div>
      <div class="biz-num"><h3>{{ converted }}</h3><p>Est. Conversions<br><small>(true positives &times; 20%)</small></p></div>
      <div class="biz-num"><h3>{{ cost_avoidance }}</h3><p>Est. Claims Savings<br><small>(conversions &times; AUD $800)</small></p></div>
    </div>
    
    <h3 style="margin-top:24px;">Sensitivity Table</h3>
    {{ sens_table }}
    
    <p class="biz-note">
      <strong>How this is calculated:</strong><br>
      1. Coverage candidates = members with risk_tier &isin; {Medium, High}, sessions_remaining_total > 0,
         and zero_allied_health_flag = 1<br>
      2. True positives = coverage candidates &times; Precision@top20% ({{ precision_pct }}%) — members genuinely at elevated acute risk<br>
      3. Conversion rate = % of true positives who book an allied health visit after nudge (adjustable — default 20%)<br>
      4. Claims savings = true positives &times; conversion rate &times; cost per avoided ED visit<br><br>
      <strong>Conservative estimate:</strong> direct ED cost only. Does not include downstream inpatient
      admissions, specialist referrals, or chronic disease management savings. Real-world savings
      are likely 2–4x higher when these are included.
    </p>
  </div>

  <h2>Model Performance</h2>
  <div class="model">
    <div class="metrics">
      <div class="metric"><h3>{{ roc_auc }}</h3><p>ROC-AUC</p></div>
      <div class="metric"><h3>{{ pr_auc }}</h3><p>PR-AUC</p></div>
      <div class="metric"><h3>{{ precision_top20 }}</h3><p>Precision @ Top 20%</p></div>
      <div class="metric"><h3>{{ recall_top20 }}</h3><p>Recall @ Top 20%</p></div>
    </div>
    <p style="font-size:13px;color:#636e72;margin-top:12px;">
      LightGBM gradient boosting binary classifier predicting acute care event risk
      (ED or inpatient admission) from claims history, benefit utilisation, and condition profile.
      Trained on 30,000 members, evaluated on 10,000 held-out members.
      <strong>Note:</strong> proof of concept on synthetic data. Val AUC ceiling ~0.727 reflects
      synthetic label construction — significantly higher AUC expected on real claims data.
    </p>
  </div>
</div>
<footer>Allied Health Coverage Analysis &middot; Proof of concept &middot; Synthetic data only &middot; Not for clinical or commercial use</footer>
</body></html>
"""

# ── Helpers ───────────────────────────────────────────────────────────────────
def fmt_num(n): return f"{int(n):,}"

# ── Build sensitivity table HTML ──────────────────────────────────────────────
sens_html = '<table><thead><tr><th>Conversion Rate</th>'
for ec in ed_costs:
    sens_html += f'<th>AUD ${ec}</th>'
sens_html += '</tr></thead><tbody>'
for _, row in sens_df.iterrows():
    sens_html += '<tr>'
    sens_html += f'<td>{row["Conversion Rate"]}</td>'
    for ec in ed_costs:
        sens_html += f'<td>{row[f"AUD ${ec}"]}</td>'
    sens_html += '</tr>'
sens_html += '</tbody></table>'

# ── Collect charts ─────────────────────────────────────────────────────────────
charts = {
    'tier':    chart_tier,
    'dist':    chart_dist,
    'cluster': chart_cluster,
    'modality': chart_mod,
    'shap':    chart_shap,
    'ct':      chart_ct,
}

# ── Render ─────────────────────────────────────────────────────────────────────
env      = Environment()
template = env.from_string(HTML_TEMPLATE)

html = template.render(
    total_scored     = fmt_num(total),
    high_risk        = fmt_num(high_risk),
    nudge_count      = fmt_num(nudge_count),
    nudge_rate       = f"{nudge_rate:.1%}",
    cost_avoidance   = f"${cost_avoidance:,}",
    true_positives   = fmt_num(true_pos),
    converted        = fmt_num(conversions),
    precision_pct    = f"{precision*100:.1f}",
    protection_ratio = protection_ratio,
    ndcg_5           = f"{summary.get('ndcg_at_5pct', 0.533):.3f}",
    roc_auc          = metrics['roc_auc'],
    pr_auc           = metrics['pr_auc'],
    precision_top20  = metrics['precision_top20pct'],
    recall_top20     = metrics['recall_top20pct'],
    causal_chain_svg = causal_chain_svg,
    charts           = charts,
    sens_table       = sens_html,
)

report_path = outputs_dir / 'report.html'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(html)
print(f"Saved: {report_path}  ({report_path.stat().st_size / 1024:.0f} KB)")
print("\n✅ Report generation complete")

causal_chain.svg not found — causal chain section will be text-only
Saved: /home/alex/personal_projects/allied-health-nudge/outputs/report.html  (719 KB)

✅ Report generation complete
